> **Outputs cleared 2026-08-15.** These notebooks are exploratory views onto the
> pipeline, not a source of paper numbers. Their stored outputs came from the
> pre-audit run, so they were cleared rather than left to look authoritative.
> Run the cells yourself against the current `results/` CSVs; the numbers quoted
> in the manuscript come from `paper/notes/REWRITE_LEDGER.md` and the result
> tables it cites, never from here.


# 01 — Data and sample definition

CSR RL0603 mascons aggregated to HydroSHEDS+Mascon L3 basins. This notebook presents the
sample: which basins are in, which are excluded and why, and what the raw series look like.

Inputs are built by `scripts/build_basin_series.py`; nothing here recomputes them.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
long_df = pd.read_csv(ROOT / "data/processed/basin_month_twsa_global.csv", parse_dates=["date"])
meta = pd.read_csv(ROOT / "data/processed/basin_meta.csv")
print(f"{meta.shape[0]} basins, {long_df['date'].nunique()} months, "
      f"{long_df['date'].min():%Y-%m} to {long_df['date'].max():%Y-%m}")

In [ ]:
# Sample definition: every exclusion is one auditable filter
print(meta["exclude_reason"].value_counts().to_string())
keep = meta[meta["exclude_reason"] == "keep"]
print(f"\nhydrology sample: {len(keep)}")
print(keep["continent"].value_counts().to_string())
print(f"\nbelow GRACE resolution (<90k km2): {keep['below_resolution'].sum()}")
print(f"glaciated stratum: {keep['glaciated'].sum()}")

In [ ]:
# Basin map colored by continent, sized by area
fig, ax = plt.subplots(figsize=(11, 5))
for cont, grp in keep.groupby("continent"):
    ax.scatter(grp["centroid_lon"], grp["centroid_lat"], s=np.sqrt(grp["area_km2"]) / 20,
               alpha=0.6, label=cont)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Hydrology sample: 234 L3 basins (marker size ~ sqrt area)")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.tight_layout()

In [ ]:
# Example series: strong seasonal (Zambezi), moderate (Congo), noise-floor (Sahara)
wide = long_df.pivot(index="date", columns="name", values="twsa_cm")
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, b in zip(axes, ["R_Zambezi_River", "R_Congo_River", "E_Sahara"]):
    ax.plot(wide.index, wide[b], lw=0.9)
    ax.set_title(b, fontsize=9)
    ax.set_ylabel("cm EWH")
    ax.axvspan(pd.Timestamp("2017-07-01"), pd.Timestamp("2018-05-31"), color="gray", alpha=0.25)
axes[0].annotate("mission gap", xy=(pd.Timestamp("2017-08-01"), axes[0].get_ylim()[1] * 0.8), fontsize=8)
plt.tight_layout()

**Notes for the paper**
- 35 missing months total: the Jul 2017 – May 2018 mission gap plus scattered 1–2 month battery gaps.
- Two duplicate monthly solutions (2011-10, 2015-04) averaged, matching the prior Africa pipeline.
- Ice sheets (27 Antarctic + 19 Greenland GIAG) and 4 water bodies are excluded up front;
  Arctic-island glaciated basins stay in but form their own reporting stratum.